# Sentiment Analysis: Exploratory Data Analysis & Preprocessing

This notebook covers the initial phase of our sentiment analysis project. We will explore the IMDB dataset, understand its distribution, and perform rigorous text preprocessing to prepare it for machine learning models.

## Objectives:
1.  **Data Loading & Inspection**: Load the dataset and check for missing values or imbalances.
2.  **Exploratory Data Analysis (EDA)**: Visualize class distribution and text characteristics.
3.  **Text Preprocessing**: Implement a robust cleaning pipeline (HTML removal, cleaning, lemmatization).
4.  **Data Export**: Save the processed data for the training phase.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from wordcloud import WordCloud
from sklearn.model_selection import train_test_split

# Settings for visualization
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Download NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

## 1. Load Dataset

We use the IMDB dataset provided in the specified local path.

In [ ]:
DATA_PATH = r"E:\Test AI Engineer Akarsa Dataset\IMDB Dataset.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dataset not found at {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Exploratory Data Analysis (EDA)

### 2.1 Class Distribution
It's crucial to check if our dataset is balanced.

In [ ]:
sns.countplot(x='sentiment', data=df, palette='viridis')
plt.title('Distribution of Sentiments')
plt.show()

print("Value Counts:")
print(df['sentiment'].value_counts())

### 2.2 Text Length Analysis
Do positive reviews tend to be longer than negative ones?

In [ ]:
df['review_length'] = df['review'].apply(len)

sns.histplot(data=df, x='review_length', hue='sentiment', kde=True, bins=50)
plt.title('Review Length Distribution by Sentiment')
plt.xlabel('Length (characters)')
plt.show()

## 3. Detailed Preprocessing

We need to clean the text from HTML tags, special characters, and convert to lowercase. We also apply lemmatization to reduce words to their base form.

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove non-alphabetic characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Convert to lowercase
    text = text.lower().strip()
    # Tokenize and remove stopwords + lemmatize
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    
    return " ".join(tokens)

print("Testing cleaning function...")
sample_text = "This is a <b>great</b> movie! I loved it, didn't you?"
print(f"Original: {sample_text}")
print(f"Cleaned:  {clean_text(sample_text)}")

In [ ]:
# Apply cleaning (this might take a minute for 50k rows)
print("Starting text cleaning...")
df['cleaned_review'] = df['review'].apply(clean_text)
print("Text cleaning complete.")
df[['review', 'cleaned_review']].head()

### 3.1 Visualization of Frequent Words
Using WordCloud to see the most frequent terms in positive and negative reviews.

In [ ]:
def show_wordcloud(data, title):
    wordcloud = WordCloud(
        background_color='white',
        max_words=200,
        max_font_size=40, 
        scale=3,
        random_state=42
    ).generate(str(data))

    plt.figure(figsize=(15, 8))
    plt.imshow(wordcloud)
    plt.axis('off')
    plt.title(title, fontsize=20)
    plt.show()

show_wordcloud(df[df['sentiment'] == 'positive']['cleaned_review'], "Positive Reviews WordCloud")
show_wordcloud(df[df['sentiment'] == 'negative']['cleaned_review'], "Negative Reviews WordCloud")

## 4. Export Prepared Data

We split the data and save it for the modeling stage.

In [ ]:
X = df['cleaned_review']
y = df['sentiment'].apply(lambda x: 1 if x == 'positive' else 0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train size: {len(X_train)}")
print(f"Test size:  {len(X_test)}")

# Save processed data (optional, but good practice)
processed_data_dir = "data/processed"
os.makedirs(processed_data_dir, exist_ok=True)

train_df = pd.DataFrame({'review': X_train, 'sentiment': y_train})
test_df = pd.DataFrame({'review': X_test, 'sentiment': y_test})

train_df.to_csv(os.path.join(processed_data_dir, "train.csv"), index=False)
test_df.to_csv(os.path.join(processed_data_dir, "test.csv"), index=False)

print(f"Data saved to {processed_data_dir}")